# 03 — Parse BOM

| | |
|---|---|
| **Pipeline stage** | `parse_bom` |
| **Service module** | `pipeline.parse_bom_only` → `parsers.upload.parse_upload_bytes` |
| **Website equivalent** | `parse_bom` in import stream; `POST /api/import/file` for direct uploads |
| **Prev / Next** | `02_extract_bom.ipynb` → `04_match_mcmaster.ipynb` |

Parses uploaded BOM files (CSV, TSV, XLSX, JSON, Markdown, HTML, text). Embedded BOMs skip this stage on the website (see `pipeline.parts_from_scrape_result`).


Regression fixture: `tests/fixtures/makerworld_export_bom.csv` (Amount/Component/Details columns). XLSX variants covered in `tests/test_spreadsheet_regression.py`.


In [ ]:
from backend.notebook_utils import (
    cache_parts,
    load_cached_parts,
    load_local_bom_file,
    notebook_progress,
    parts_to_dataframe,
    prepare_crawl_env,
    resolve_parts_offline,
)
from backend.services.pipeline import parse_bom_only
from backend.services.parsers.helpers.specification_checks import (
    check_parts_specifications,
    format_specification_issues,
)

prepare_crawl_env(reload_backend=False)
progress = notebook_progress("03")

parts = load_cached_parts()
source = "embedded cache (skipped parse on website too)"

if not parts:
    local = load_local_bom_file()
    if local:
        content, filename = local
        try:
            parts = parse_bom_only(content, filename, on_progress=progress)
            source = f"parse_bom_only({filename})"
        except Exception as exc:
            print(f"Parse failed: {exc}")

if not parts:
    parts, source = await resolve_parts_offline()
    if parts:
        print("Offline fallback — not the live parse path")

if parts:
    cache_parts(parts)
    spec_issues = check_parts_specifications(parts)
    if spec_issues:
        print(format_specification_issues(spec_issues))
    else:
        print(f"Specification check OK ({len(parts)} parts)")
    print(f"{len(parts)} parts from {source}")
    parts_to_dataframe(parts)
else:
    print("No parts — run 02 or add data/sample_bom.csv")
